# Week 9 · Day 2 — The Limits AND Superpowers of Transfer Learning
## See the features · measure domain shift · turn the unfreeze knob · zero-shot with CLIP

**Time:** ~20 minutes · **Four demos, one per big idea from the lecture:**

1. **See the features** — visualize what a frozen backbone *actually* computes: early layers = edges, deep layers = abstract parts.
2. **Close vs. far domain** — transfer the *same* frozen backbone to CIFAR-10 (natural photos, **close** to ImageNet) and Fashion-MNIST (grayscale clothing, **far**), and measure what the backbone truly earns.
3. **The unfreeze knob** — freeze-all vs. unfreeze-the-last-block + low-LR fine-tune, and watch the tradeoff.
4. **The superpower** — CLIP classifies an image against text labels with **zero training**.

> Same backbone you drove yesterday (**MobileNetV2**). Yesterday was *how* to freeze and fine-tune.
> Today is *when* it works and when it breaks — so you can **predict** transfer, not just run it.

**Heads-up:** CIFAR-10 and Fashion-MNIST ship with Keras (no download). The CLIP demo downloads one
small model the first time (~600 MB) and is wrapped in `try/except`, so a failed download prints a
message instead of crashing the notebook.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
from keras import layers
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input
from keras.datasets import cifar10, fashion_mnist
import numpy as np
import matplotlib.pyplot as plt

keras.utils.set_random_seed(42)   # reproducible
print('keras', keras.__version__, '| backend:', keras.backend.backend())

---
## Demo 1 · See what pretrained features actually are

Yesterday Dr. Gray *drew* the feature hierarchy on the board — **edges → textures → parts → whole
objects** — and asked you to trust it. Let's **look** at the real thing. We tap an **early** layer and a
**deep** layer of the frozen MobileNetV2 and plot the feature maps for one image.

In [ ]:
# The same backbone you used yesterday: MobileNetV2, ImageNet weights, no classifier head.
base = MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights='imagenet')
print(f'{len(base.layers)} layers, {base.count_params():,} parameters -- all pretrained on ImageNet.')

# Grab one natural photo to look at (an automobile from CIFAR-10), upsized to 160x160.
(x_cifar, y_cifar), _ = cifar10.load_data()
one = x_cifar[np.where(y_cifar.ravel() == 1)[0][0]]                      # label 1 = automobile
one_160 = keras.ops.convert_to_numpy(
    keras.ops.image.resize(one[None].astype('float32'), (160, 160)))
plt.figure(figsize=(3, 3)); plt.imshow(one_160[0].astype('uint8'))
plt.axis('off'); plt.title('input image'); plt.show()

In [ ]:
# Peek inside: a mini-model that outputs the activations of ONE chosen layer, then plot its maps.
def show_feature_maps(layer_name, n=12):
    # TODO (1): tap one layer -- build a keras.Model from base.input to that layer's .output
    #           hint: base.get_layer(layer_name).output
    probe = ____
    acts = probe.predict(preprocess_input(one_160.copy()), verbose=0)[0]  # (H, W, C)
    fig, axes = plt.subplots(2, n // 2, figsize=(n, 3.2))
    for i, ax in enumerate(axes.flat):
        ax.imshow(acts[:, :, i], cmap='viridis'); ax.axis('off')
    fig.suptitle(f'{layer_name}  ->  maps are {acts.shape[0]}x{acts.shape[1]}, {acts.shape[2]} channels')
    plt.tight_layout(); plt.show()

show_feature_maps('Conv1_relu')            # EARLY : 80x80 -- edges / colors / gradients
show_feature_maps('block_6_expand_relu')   # MIDDLE: 20x20 -- textures / patterns
show_feature_maps('block_13_expand_relu')  # DEEP  : 10x10 -- abstract object-parts

**Read the maps.** The **early** layer (`Conv1_relu`) has big 80x80 maps, each firing on edges,
corners, and color blobs — local, fine detail. The **deep** layer (`block_13_expand_relu`) has tiny
10x10 maps that are coarse and abstract — each is a detector for some higher-level pattern. That is
*exactly* the hierarchy from yesterday's board: generic edges at the bottom, specific parts at the top.

> **This is why features transfer.** *"An edge is an edge"* — the bottom layers are generic, so they're
> useful for **any** images. The top layers are tuned to ImageNet's 1,000 objects — which is where
> trouble starts if your images don't look like ImageNet.

**Your turn:** swap in `'block_3_expand_relu'` (earlier) or `'out_relu'` (the deepest, 5x5) and re-run.

---
## Demo 2 · Close vs. far domain — what does the backbone really earn?

Yesterday's flowers were **close** to ImageNet, so the frozen backbone shone. But what if your images
*don't* look like natural photos? We transfer the same frozen backbone to two datasets:

- **CIFAR-10** — natural color objects (cars, dogs, ships). **Close** to ImageNet.
- **Fashion-MNIST** — grayscale clothing silhouettes, upsampled to 3 channels. **Far** from ImageNet.

Same recipe as Day 1: frozen backbone -> global-average-pool -> a small `Dense` head.

In [ ]:
def to_160_rgb(imgs):
    """Resize a batch to 160x160x3 (Fashion-MNIST is grayscale -> repeat to 3 channels)."""
    if imgs.ndim == 3:                       # (N, H, W) grayscale
        imgs = np.repeat(imgs[..., None], 3, axis=-1)
    return keras.ops.convert_to_numpy(keras.ops.image.resize(imgs.astype('float32'), (160, 160)))

N_TRAIN, N_TEST = 2000, 500
(cf_xtr, cf_ytr), (cf_xte, cf_yte) = cifar10.load_data()
(fm_xtr, fm_ytr), (fm_xte, fm_yte) = fashion_mnist.load_data()

close = dict(name='CIFAR-10 (CLOSE)',
             xtr=to_160_rgb(cf_xtr[:N_TRAIN]), ytr=cf_ytr.ravel()[:N_TRAIN],
             xte=to_160_rgb(cf_xte[:N_TEST]),  yte=cf_yte.ravel()[:N_TEST])
far   = dict(name='Fashion-MNIST (FAR)',
             xtr=to_160_rgb(fm_xtr[:N_TRAIN]), ytr=fm_ytr[:N_TRAIN],
             xte=to_160_rgb(fm_xte[:N_TEST]),  yte=fm_yte[:N_TEST])

# LOOK at them -- top row: natural photos (close); bottom row: grayscale silhouettes (far).
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for j in range(6):
    axes[0, j].imshow(close['xtr'][j].astype('uint8')); axes[0, j].axis('off')
    axes[1, j].imshow(far['xtr'][j].astype('uint8'));   axes[1, j].axis('off')
fig.suptitle('Top: CIFAR-10 (close to ImageNet)      |      Bottom: Fashion-MNIST (far)')
plt.tight_layout(); plt.show()

In [ ]:
base.trainable = False   # freeze: learning stops, computing continues (yesterday's rule)

def backbone_features(imgs):
    """Run the frozen backbone once and global-average-pool -> one 1280-vector per image."""
    # TODO (2): 'match the backbone!' -- preprocess_input(imgs.copy()), run base.predict(..., verbose=0),
    #           then collapse each (5,5,1280) map to a 1280-vector with .mean(axis=(1, 2))
    maps = ____
    return ____

def train_head(features_tr, ytr, features_te, yte, epochs=10):
    keras.utils.set_random_seed(0)
    head = keras.Sequential([layers.Input((features_tr.shape[1],)),
                             layers.Dropout(0.3),
                             layers.Dense(10, activation='softmax')])
    head.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
    head.fit(features_tr, ytr, epochs=epochs, batch_size=32, verbose=0)
    return head.evaluate(features_te, yte, verbose=0)[1]

for d in (close, far):
    d['acc'] = train_head(backbone_features(d['xtr']), d['ytr'],
                          backbone_features(d['xte']), d['yte'])
    print(f"{d['name']:22s} frozen-backbone accuracy = {d['acc']:.3f}")

**Surprise:** the **far** set (Fashion-MNIST) probably scored *higher* than the close one. So is
transfer working *better* on clothing?? Before you believe that, let's run a control: skip the backbone
entirely and train the same head on tiny **16x16 pixel thumbnails**. If a dumb pixel classifier already
does well, the backbone wasn't the thing doing the work.

In [ ]:
def pixel_features(imgs):
    """No backbone at all -- just a 16x16 color thumbnail, flattened."""
    return keras.ops.convert_to_numpy(keras.ops.image.resize(imgs, (16, 16))).reshape(len(imgs), -1) / 255.

for d in (close, far):
    d['pix'] = train_head(pixel_features(d['xtr']), d['ytr'], pixel_features(d['xte']), d['yte'])
    print(f"{d['name']:22s} raw-pixel baseline = {d['pix']:.3f}   ->  backbone ADDS {100*(d['acc']-d['pix']):+.1f} points")

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(2)
ax.bar(x - 0.2, [close['pix'], far['pix']], 0.4, label='raw pixels (no backbone)', color='#c2c2c2')
ax.bar(x + 0.2, [close['acc'], far['acc']], 0.4, label='frozen ImageNet backbone', color='#2aa76a')
ax.set_xticks(x); ax.set_xticklabels(['CIFAR-10\n(close)', 'Fashion-MNIST\n(far)'])
ax.set_ylabel('test accuracy'); ax.set_ylim(0, 1); ax.legend()
ax.set_title('What did the ImageNet backbone actually earn?')
plt.tight_layout(); plt.show()

**The honest lesson (domain shift).**

- On **CIFAR-10 (close)**, raw pixels get ~28% and the backbone leaps to ~80% — the backbone adds **~50
  points**. It's doing nearly all the work, because natural photos are exactly what ImageNet features were
  built for.
- On **Fashion-MNIST (far)**, raw pixels *already* get ~80% (clothing silhouettes are trivially
  pixel-separable), and the backbone adds only **~6 points**.

> ⚠️ **The 85% on the far set is not transfer working — it's the task being easy.** That is **domain
> shift**: the further your images sit from ImageNet's natural photos, the less the pretrained features
> actually contribute. Don't trust the headline accuracy — ask *what the backbone earned.* This is why
> medical scans, satellite imagery, and line-art can't just borrow a frozen ImageNet backbone.

---
## Demo 3 · The unfreeze knob — adapting the backbone

On a far-ish domain the frozen *top* features underdeliver. Yesterday's Phase 2 answer: **unfreeze the
top of the backbone and fine-tune it at a tiny learning rate**, so those layers adapt to *your* images.
Let's turn that knob on the close set and watch the tradeoff.

In [ ]:
# Small subset so the full backbone can backprop quickly.
sub = dict(xtr=close['xtr'][:800], ytr=close['ytr'][:800],
           xte=close['xte'][:400], yte=close['yte'][:400])
xtr_p, xte_p = preprocess_input(sub['xtr'].copy()), preprocess_input(sub['xte'].copy())

keras.utils.set_random_seed(0)
base.trainable = False                         # freeze first (yesterday's rule)
inp = keras.Input((160, 160, 3))
x = base(inp, training=False)                  # training=False keeps BatchNorm in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(10, activation='softmax')(x)
model = keras.Model(inp, out)
model.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(xtr_p, sub['ytr'], epochs=3, batch_size=32, verbose=0)
acc_frozen = model.evaluate(xte_p, sub['yte'], verbose=0)[1]
print(f'FREEZE ALL (train head only):        {acc_frozen:.3f}')

# TODO (3): unfreeze ONLY the last 20 layers of the backbone (keep the rest frozen),
#           then compile with the tiny fine-tuning learning rate from Day 1 (1e-5).
base.trainable = True
for layer in ____:                             # keep the bottom ~134 layers frozen
    layer.trainable = False
model.compile(keras.optimizers.Adam(____), 'sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(xtr_p, sub['ytr'], epochs=3, batch_size=32, verbose=0)
acc_unfrozen = model.evaluate(xte_p, sub['yte'], verbose=0)[1]
print(f'UNFREEZE last 20 + fine-tune @ 1e-5:  {acc_unfrozen:.3f}   (change: {100*(acc_unfrozen-acc_frozen):+.1f} pts)')

**Read the tradeoff.** Unfreezing lets the top layers adapt — but on only 800 images the gain is
**small**, and if you push the learning rate up it can go *negative*. That's the honest catch: fine-tuning
is not free. A high LR would blow the pretrained weights out of their good valley — **catastrophic
forgetting** — which is exactly why Day 1 hammered **1e-5, 100x lower**.

That's the lecture's decision framework in one experiment:

| | **Close** domain | **Far** domain |
|---|---|---|
| **Little data** | freeze all (train head) | freeze most + augment hard |
| **Lots of data** | fine-tune top layers | unfreeze deep / train more |

**Your turn:** change `-20` to `-40` (unfreeze more), or bump `1e-5` to `1e-4`, and re-run. When does
fine-tuning start to *hurt*?

---
## Demo 4 · The superpower — CLIP zero-shot (no training at all)

Every demo so far trained *something*, even a tiny head. The superpower from yesterday's modern-CV tour:
a model that understands **images and text in one shared space**, so you classify by simply *describing*
the classes in words. **Zero training, no head.** That's **CLIP**.

In [ ]:
# CLIP: no training, no head. Describe the candidate classes in plain English; ask which one fits.
try:
    from transformers import CLIPModel, CLIPProcessor
    from PIL import Image
    import torch

    clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
    proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

    pil_img = Image.fromarray(one_160[0].astype('uint8'))     # the automobile from Demo 1
    # TODO (4): write 3-5 candidate text labels to score the image against.
    #           Include the true class (a car) and some distractors. Format: 'a photo of a ___'
    candidate_labels = ____

    inputs = proc(text=candidate_labels, images=pil_img, return_tensors='pt', padding=True)
    with torch.no_grad():
        probs = clip(**inputs).logits_per_image.softmax(dim=1)[0]

    print('CLIP zero-shot (never trained on these labels):')
    for label, p in sorted(zip(candidate_labels, probs.tolist()), key=lambda t: -t[1]):
        print(f'  {p*100:5.1f}%  {label}')
except Exception as e:
    print('CLIP demo skipped (model could not be downloaded in this environment):')
    print('  ', type(e).__name__, str(e)[:150])
    print('That is fine -- the concept is: CLIP scores an image against text labels with zero training.')

---
## Wrap-up · one mental model: **backbone + head**

| System | Pretrained **backbone** | Task-specific **head** | What you train |
|---|---|---|---|
| Yesterday's flowers | MobileNetV2 (ImageNet) | GAP -> Dense(5) | the head (+ optional top layers) |
| Object detection (YOLO) | a CNN backbone | box + label head | the head, then fine-tune |
| **Next week** — text | a Transformer (DistilBERT) | classification head | the head — *same `from_pretrained` move* |
| CLIP zero-shot | image encoder + text encoder | none — just your text labels | **nothing** |

> **The whole two weeks in one line:** download a **backbone** that already learned to see (or, next week,
> to read), bolt on a small **head** for your job, and adapt only as much as your data and your domain
> distance allow. Transfer learning isn't magic — once you know how far your problem sits from the
> pretraining data, you can *predict* whether it will work.

### Reflection (answer in a sentence each)
1. Name a dataset where a *frozen* ImageNet backbone would barely help, and explain why using the
   **feature hierarchy** (which layers stop being useful, and why).
2. You have **300** labeled chest X-rays (**far** from ImageNet). Using the two-dial framework
   (data size x domain distance), how much of the backbone would you unfreeze — and what is the danger
   if you set the learning rate too high?
3. What single idea connects **this** week (images) and **next** week (text)?